# 第 12 章 アンサンブル学習

弱い学習器を集めて強い学習器を作ります。多数決（バギング）と逐次的な重み付け（AdaBoost）を比べます。

対応する記事: [第 12 章 アンサンブル学習（Jupyter Notebook（Python） の言語版）](../../../docs/article/grokking-machine-learning/python/ch12.md)

実装本体: `apps/grokking-ml-python/src/`

## セットアップ

実装本体（`../src/grokking_ml/`）を読み込みます。**ノートブックにコードを複製せず、記事と同じ実装をそのまま使います。**

```bash
cd apps/grokking-ml-python
uv sync
uv run jupyter lab notebooks/
```

In [1]:
import pathlib
import sys

sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))

from grokking_ml.ch09_decision_trees import *
from grokking_ml.ch12_ensembles import *

## 切り株 1 本では解けないデータ

1 次元上に 3 つの領域が並んでいます。深さ 1 の木（切り株）は **線を 1 本しか引けない** ので、3 領域は分けられません。

In [2]:
points = [(float(i), 1.0) for i in range(1, 9)]
labels = [1, 1, -1, -1, -1, -1, 1, 1]

stump = build_tree(points, labels, max_depth=1)
print(stump)
print(f"切り株 1 本の正解率 {tree_accuracy(stump, points, labels):.2f}")

Node(split=Split(feature=0, threshold=2.5), left=Leaf(label=1), right=Leaf(label=-1))
切り株 1 本の正解率 0.75


## バギングと AdaBoost を比べる

**同じ弱学習器を同じ本数使っても、束ね方で結果が分かれます。**

バギングが効かないのは、復元抽出でデータを揺らしても **どの標本でも「x = 2.5 で切る」のが最良** なので、10 本ともほぼ同じ木になるからです。同じ意見を 10 回聞いても結論は変わりません。

In [3]:
forest = train_forest(points, labels, tree_count=10, max_depth=1)
boosted = train_adaboost(points, labels, rounds=10, max_depth=1)

print(f"切り株 1 本            {tree_accuracy(stump, points, labels):.2f}")
print(f"バギング（10 本）      {accuracy(forest, points, labels):.2f}")
print(f"AdaBoost（10 本）      {accuracy(boosted, points, labels):.2f}")

切り株 1 本            0.75
バギング（10 本）      0.75
AdaBoost（10 本）      1.00


## AdaBoost が生む役割分担

**1 本目は左端、2 本目は右端に注目しました。** 1 本目が右端を外したので、その点の重みが上がり、2 本目が拾いに行ったのです。誰も指示していないのに役割分担が生まれます。

In [4]:
for index, learner in enumerate(boosted.learners[:4], start=1):
    print(f"ラウンド {index}  発言権 {learner.weight:.4f}")
    print(f"          {learner.tree}")

ラウンド 1  発言権 0.5493
          Node(split=Split(feature=0, threshold=2.5), left=Leaf(label=1), right=Leaf(label=-1))
ラウンド 2  発言権 0.8047
          Node(split=Split(feature=0, threshold=6.5), left=Leaf(label=-1), right=Leaf(label=1))
ラウンド 3  発言権 0.6931
          Node(split=Split(feature=0, threshold=2.5), left=Leaf(label=1), right=Leaf(label=1))
ラウンド 4  発言権 0.7332
          Node(split=Split(feature=0, threshold=2.5), left=Leaf(label=1), right=Leaf(label=-1))


## 発言権の式

**誤り率 0.5 でちょうど 0 になります。** 当てずっぽうの意見は無視されます。0.5 より悪い学習器は「逆を言えば当たる」ので負の発言権になります。

In [5]:
print(f"{'誤り率':>8} {'発言権':>10}")
for error in [0.0, 0.05, 0.2, 0.4, 0.5, 0.7]:
    print(f"{error:>8.2f} {learner_weight(error):>10.4f}")

     誤り率        発言権
    0.00    11.5129
    0.05     1.4722
    0.20     0.6931
    0.40     0.2027
    0.50     0.0000
    0.70    -0.4236


## 試してみる: ラウンド数を変える

In [6]:
for rounds in [1, 2, 3, 5, 10]:
    m = train_adaboost(points, labels, rounds=rounds, max_depth=1)
    print(f"{rounds:>3} ラウンド  学習器 {len(m.learners):>2} 本  "
          f"正解率 {accuracy(m, points, labels):.2f}")

  1 ラウンド  学習器  1 本  正解率 0.75
  2 ラウンド  学習器  2 本  正解率 0.75
  3 ラウンド  学習器  3 本  正解率 1.00
  5 ラウンド  学習器  5 本  正解率 1.00
 10 ラウンド  学習器 10 本  正解率 1.00
